##

##  Generative AI for Tabular PCOS Data — Full Benchmark

In [1]:
   !pip install pandas numpy scikit-learn imbalanced-learn ctgan torch
   ! pip install xgboost

In [3]:
"""
=============================================================================
 Generative AI for Tabular PCOS Data — Full Benchmark
=============================================================================
 Compares synthetic-data oversamplers on two PCOS datasets:

   Classical  : SMOTE, ADASYN, BorderlineSMOTE
   GAN        : CTGAN, CTAB-GAN+ (adapter)
   VAE        : TVAE
   Diffusion  : TabDDPM-lite  (compact re-implementation)
   Latent Dif.: TabSyn-lite   (VAE -> latent diffusion, compact)

 Protocol (this is the part examiners check):
   1. Stratified split of the REAL data  -> real_train / real_test
   2. Generator is fitted on real_train ONLY
   3. Synthetic minority rows are added to real_train
   4. Classifier trained on the augmented train set
   5. Scored on real_test  -- NEVER on synthetic rows

 Install:
   pip install pandas numpy scikit-learn imbalanced-learn ctgan torch
   pip install xgboost              # optional
=============================================================================
"""

from __future__ import annotations

import argparse
import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


# =============================================================================
# 1. DATA LOADING + CLEANING
# =============================================================================

def load_clinical(path: str) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """Kaggle-style clinical PCOS file (541 x 45). Returns X, y, categorical cols."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    # Drop identifiers and the all-empty trailing column
    for junk in ["Sl. No", "Patient File No.", "Unnamed: 44"]:
        if junk in df.columns:
            df = df.drop(columns=junk)
    df = df.dropna(axis=1, how="all")

    target = "PCOS (Y/N)"
    y = pd.to_numeric(df[target], errors="coerce")
    X = df.drop(columns=target)

    # These two columns carry dirty strings ("1.99.", "a") -> coerce to NaN
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    # Binary Y/N flags + Blood Group behave as categorical for the generators
    cat_cols = [c for c in X.columns if X[c].dropna().nunique() <= 2]
    if "Blood Group" in X.columns:
        cat_cols.append("Blood Group")

    keep = y.notna()
    X, y = X.loc[keep], y.loc[keep].astype(int)

    # Median impute (fit on the whole frame here is fine: no target leakage)
    X = X.fillna(X.median(numeric_only=True))
    return X.reset_index(drop=True), y.reset_index(drop=True), cat_cols


def load_survey(path: str) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """Lifestyle/survey PCOS file (173 x 36). Returns X, y, categorical cols."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    # Free-text column -- unusable as a feature, drop it
    df = df.drop(columns=[c for c in ["PCOS_Medication"] if c in df.columns])

    # Target: keep only unambiguous Yes/No, drop the "not diagnosed by a doctor" row
    df["PCOS"] = df["PCOS"].astype(str).str.strip()
    df = df[df["PCOS"].isin(["Yes", "No"])].copy()
    y = (df["PCOS"] == "Yes").astype(int)
    X = df.drop(columns="PCOS")

    # Ordinal encodings that preserve real ordering (better than one-hot here)
    ordinal_maps = {
        "Age": {"Less than 20": 0, "20-25": 1, "25-30": 2,
                "30-35": 3, "35-44": 4, "45 and above": 5},
        "Exercise_Frequency": {"Never": 0, "Rarely": 1, "1-2 Times a Week": 2,
                               "3-4 Times a Week": 3, "Daily": 4},
        "Exercise_Duration": {"Not Applicable": 0, "Less than 30 minutes": 1,
                              "30 minutes": 2, "More than 30 minutes": 3,
                              "30 minutes to 1 hour": 4},
        "Sleep_Hours": {"Less than 6 hours": 0, "6-8 hours": 1,
                        "9-12 hours": 2, "More than 12 hours": 3},
        "Exercise_Benefit": {"Not at All": 0, "Not Much": 1,
                             "Somewhat": 2, "Yes Significantly": 3},
    }
    for col, mapping in ordinal_maps.items():
        if col in X.columns:
            X[col] = X[col].astype(str).str.strip().map(mapping)

    # Everything still object -> label-encode as nominal categorical
    cat_cols: list[str] = []
    for col in X.columns:
        if X[col].dtype == object or str(X[col].dtype) == "str":
            X[col] = X[col].astype(str).str.strip()
            X[col] = pd.factorize(X[col])[0]
            cat_cols.append(col)
        elif X[col].dropna().nunique() <= 2:
            cat_cols.append(col)

    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    return X.reset_index(drop=True), y.reset_index(drop=True), cat_cols


# =============================================================================
# 2. GENERATOR INTERFACE
# =============================================================================

@dataclass
class SynthResult:
    """Synthetic rows produced by one generator."""
    name: str
    X: pd.DataFrame
    y: pd.Series
    seconds: float = 0.0
    note: str = ""


class BaseGenerator:
    """All generators implement: fit(X_train, y_train) -> sample(n_minority)."""
    name = "base"
    requires_torch = False

    def __init__(self, cat_cols: list[str], seed: int = SEED):
        self.cat_cols = cat_cols
        self.seed = seed

    def fit_sample(self, X: pd.DataFrame, y: pd.Series, n_needed: int) -> pd.DataFrame:
        raise NotImplementedError


# ---------------------------------------------------------------- classical --
class ImblearnGenerator(BaseGenerator):
    """SMOTE / ADASYN / BorderlineSMOTE. Interpolates, does not learn a density."""

    def __init__(self, kind: str, cat_cols, seed=SEED):
        super().__init__(cat_cols, seed)
        self.kind = kind
        self.name = kind

    def fit_sample(self, X, y, n_needed):
        from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE

        # k must be < number of minority samples
        k = int(min(5, max(1, (y == 1).sum() - 1)))
        algos = {
            "SMOTE": SMOTE(random_state=self.seed, k_neighbors=k),
            "ADASYN": ADASYN(random_state=self.seed, n_neighbors=k),
            "BorderlineSMOTE": BorderlineSMOTE(random_state=self.seed, k_neighbors=k),
        }
        Xr, yr = algos[self.kind].fit_resample(X, y)

        # Rows appended past the original length are the synthetic ones
        synth = Xr.iloc[len(X):].copy()
        synth = self._round_categoricals(synth)
        return synth.head(n_needed)

    def _round_categoricals(self, df):
        for c in self.cat_cols:
            if c in df.columns:
                df[c] = df[c].round().astype(int)
        return df


# ------------------------------------------------------------ GAN and VAE ----
class CTGANGenerator(BaseGenerator):
    """Conditional Tabular GAN (Xu et al. 2019) via the `ctgan` package."""
    name = "CTGAN"

    def __init__(self, cat_cols, seed=SEED, epochs=300):
        super().__init__(cat_cols, seed)
        self.epochs = epochs

    def fit_sample(self, X, y, n_needed):
        from ctgan import CTGAN

        train = X.copy()
        train["__label__"] = y.values
        discrete = [c for c in self.cat_cols if c in train.columns] + ["__label__"]

        self._collapsed = False
        model = CTGAN(epochs=self.epochs, verbose=False, cuda=False)
        model.fit(train, discrete_columns=discrete)
        synth = self._sample_minority(model, n_needed, train.columns)
        if self._collapsed:
            self.name += " [minority-only fallback]"
            synth = self._fit_minority_only(CTGAN, X, y, n_needed)
        return synth

    def _sample_minority(self, model, n_needed, columns):
        """
        Oversample a pool, then keep only minority rows.

        If the model never emits the minority label (mode collapse, which GANs
        and VAEs do on small imbalanced tables), fall back to refitting on the
        minority subset alone rather than silently returning zero rows.
        """
        pool, tries, total = [], 0, 0
        while total < n_needed and tries < 12:
            batch = model.sample(max(500, n_needed * 6))
            lab = pd.to_numeric(batch["__label__"], errors="coerce").round()
            keep = batch[lab == 1]
            if len(keep):
                pool.append(keep)
                total += len(keep)
            tries += 1

        if total == 0:
            self._collapsed = True
            return pd.DataFrame(columns=[c for c in columns if c != "__label__"])

        synth = pd.concat(pool, ignore_index=True).drop(columns="__label__", errors="ignore")
        return synth.head(n_needed)

    def _fit_minority_only(self, model_cls, X, y, n_needed):
        """Fallback path: learn the minority class distribution directly."""
        minority = X[y == 1].copy()
        discrete = [c for c in self.cat_cols if c in minority.columns]
        model = model_cls(epochs=self.epochs, cuda=False)
        model.fit(minority, discrete_columns=discrete)
        return model.sample(n_needed)


class TVAEGenerator(CTGANGenerator):
    """Tabular VAE (same paper as CTGAN, variational instead of adversarial)."""
    name = "TVAE"

    def fit_sample(self, X, y, n_needed):
        from ctgan import TVAE

        train = X.copy()
        train["__label__"] = y.values
        discrete = [c for c in self.cat_cols if c in train.columns] + ["__label__"]

        self._collapsed = False
        model = TVAE(epochs=self.epochs, cuda=False)
        model.fit(train, discrete_columns=discrete)
        synth = self._sample_minority(model, n_needed, train.columns)
        if self._collapsed:
            self.name += " [minority-only fallback]"
            synth = self._fit_minority_only(TVAE, X, y, n_needed)
        return synth


class CTABGANPlusGenerator(CTGANGenerator):
    """
    CTAB-GAN+ adapter.

    CTAB-GAN+ is NOT on PyPI -- it ships as a research repo:
        git clone https://github.com/Team-TUD/CTAB-GAN-Plus
    Point CTABGAN_REPO at the clone and this wraps it. Without the repo the
    benchmark falls back to CTGAN and labels the row accordingly, so you are
    never silently reporting a number that came from a different model.
    """
    name = "CTAB-GAN+"
    CTABGAN_REPO = None  # e.g. "/home/you/CTAB-GAN-Plus"

    def fit_sample(self, X, y, n_needed):
        if self.CTABGAN_REPO is None:
            self.name = "CTAB-GAN+ (unavailable -> CTGAN fallback)"
            return super().fit_sample(X, y, n_needed)

        import sys
        sys.path.insert(0, self.CTABGAN_REPO)
        from model.ctabgan import CTABGAN  # type: ignore

        train = X.copy()
        train["__label__"] = y.values
        tmp = "_ctabgan_input.csv"
        train.to_csv(tmp, index=False)

        model = CTABGAN(
            raw_csv_path=tmp,
            categorical_columns=[c for c in self.cat_cols if c in train.columns],
            problem_type={"Classification": "__label__"},
            epochs=self.epochs,
        )
        model.fit()
        pool = model.generate_samples(max(500, n_needed * 6))
        synth = pool[pool["__label__"] == 1].drop(columns="__label__")
        return synth.head(n_needed)


# ------------------------------------------------------- diffusion (torch) ---
class TabDDPMLite(BaseGenerator):
    """
    Compact class-conditional Gaussian diffusion for tabular data.

    Follows the TabDDPM recipe (Kotelnikov et al. 2023) in spirit:
      * numeric features -> quantile-normalised to ~N(0,1)
      * categorical      -> one-hot, then treated in the same Gaussian space
      * denoiser         -> MLP conditioned on timestep embedding + class label
      * sampling         -> ancestral DDPM sampling with y fixed to the minority class

    This is a re-implementation, not the official repo (yandex-research/tab-ddpm).
    Say so in your write-up.
    """
    name = "TabDDPM-lite"
    requires_torch = True

    def __init__(self, cat_cols, seed=SEED, epochs=4000, timesteps=200,
                 hidden=256, lr=1e-3):
        super().__init__(cat_cols, seed)
        self.epochs, self.timesteps = epochs, timesteps
        self.hidden, self.lr = hidden, lr

    # -- preprocessing -------------------------------------------------------
    def _build_transform(self, X):
        from sklearn.preprocessing import QuantileTransformer, OneHotEncoder

        self.num_cols = [c for c in X.columns if c not in self.cat_cols]
        self.cat_cols_ = [c for c in self.cat_cols if c in X.columns]

        self.qt = QuantileTransformer(
            output_distribution="normal",
            n_quantiles=max(10, min(1000, len(X) // 2)),
            random_state=self.seed,
        )
        num = self.qt.fit_transform(X[self.num_cols]) if self.num_cols else np.zeros((len(X), 0))

        if self.cat_cols_:
            self.ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            cat = self.ohe.fit_transform(X[self.cat_cols_].astype(str))
            self.cat_sizes = [len(c) for c in self.ohe.categories_]
        else:
            self.ohe, cat, self.cat_sizes = None, np.zeros((len(X), 0)), []

        return np.hstack([num, cat]).astype(np.float32)

    def _inverse_transform(self, Z):
        n_num = len(self.num_cols)
        out = {}

        if n_num:
            num = self.qt.inverse_transform(Z[:, :n_num])
            for i, c in enumerate(self.num_cols):
                out[c] = num[:, i]

        # Each categorical block -> argmax -> original level
        offset = n_num
        for size, col, cats in zip(self.cat_sizes, self.cat_cols_, self.ohe.categories_ if self.ohe else []):
            block = Z[:, offset:offset + size]
            out[col] = np.asarray(cats)[block.argmax(axis=1)]
            offset += size

        df = pd.DataFrame(out)
        for c in self.cat_cols_:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        return df

    # -- model ---------------------------------------------------------------
    def _make_net(self, d_in, n_classes):
        import torch.nn as nn

        class Denoiser(nn.Module):
            def __init__(self, d_in, hidden, n_classes, t_dim=64):
                super().__init__()
                self.t_dim = t_dim
                self.label_emb = nn.Embedding(n_classes, t_dim)
                self.net = nn.Sequential(
                    nn.Linear(d_in + t_dim * 2, hidden), nn.SiLU(),
                    nn.Linear(hidden, hidden), nn.SiLU(),
                    nn.Linear(hidden, hidden), nn.SiLU(),
                    nn.Linear(hidden, d_in),
                )

            def timestep_embedding(self, t):
                import torch
                half = self.t_dim // 2
                freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / half)
                args = t.float()[:, None] * freqs[None]
                return torch.cat([args.cos(), args.sin()], dim=-1)

            def forward(self, x, t, y):
                import torch
                h = torch.cat([x, self.timestep_embedding(t), self.label_emb(y)], dim=-1)
                return self.net(h)

        return Denoiser(d_in, self.hidden, n_classes)

    def fit_sample(self, X, y, n_needed):
        import torch

        torch.manual_seed(self.seed)
        Z = self._build_transform(X)
        y_arr = y.values.astype(np.int64)

        # Cosine noise schedule (Nichol & Dhariwal) -- more stable than linear
        s, T = 0.008, self.timesteps
        steps = np.arange(T + 1) / T
        alphas_cum = np.cos((steps + s) / (1 + s) * np.pi / 2) ** 2
        alphas_cum = alphas_cum / alphas_cum[0]
        betas = np.clip(1 - alphas_cum[1:] / alphas_cum[:-1], 1e-8, 0.999)

        betas_t = torch.tensor(betas, dtype=torch.float32)
        alphas_t = 1.0 - betas_t
        acp = torch.cumprod(alphas_t, dim=0)

        Zt = torch.tensor(Z)
        yt = torch.tensor(y_arr)
        net = self._make_net(Z.shape[1], int(y_arr.max()) + 1)
        opt = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=1e-5)

        batch = min(256, len(Zt))
        for _ in range(self.epochs):
            idx = torch.randint(0, len(Zt), (batch,))
            x0, yb = Zt[idx], yt[idx]
            t = torch.randint(0, T, (batch,))
            noise = torch.randn_like(x0)
            a = acp[t][:, None]
            xt = a.sqrt() * x0 + (1 - a).sqrt() * noise

            loss = ((net(xt, t, yb) - noise) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()

        net.eval()
        x = self._ddpm_sample(net, n_needed, Z.shape[1], alphas_t, acp, betas_t, T)
        return self._inverse_transform(x)

    # -- sampler -------------------------------------------------------------
    @staticmethod
    def _ddpm_sample(net, n, d, alphas_t, acp, betas_t, T, clamp=4.0):
        """
        Ancestral DDPM sampling with clamped x0 prediction.

        Predicting x0 and clipping it to the plausible range of the normalised
        space, then using the true posterior variance (beta_tilde), keeps the
        chain from blowing up. Without this, small tabular datasets produce
        synthetic rows with 2-3x the real standard deviation.
        """
        import torch

        with torch.no_grad():
            x = torch.randn(n, d)
            y_gen = torch.ones(n, dtype=torch.long)
            acp_prev = torch.cat([torch.ones(1), acp[:-1]])

            for t in reversed(range(T)):
                tt = torch.full((n,), t, dtype=torch.long)
                eps = net(x, tt, y_gen)

                x0 = (x - (1 - acp[t]).sqrt() * eps) / acp[t].sqrt()
                x0 = x0.clamp(-clamp, clamp)

                mean = (acp_prev[t].sqrt() * betas_t[t] / (1 - acp[t]) * x0
                        + alphas_t[t].sqrt() * (1 - acp_prev[t]) / (1 - acp[t]) * x)

                if t == 0:
                    x = mean
                else:
                    var = betas_t[t] * (1 - acp_prev[t]) / (1 - acp[t])
                    x = mean + var.sqrt() * torch.randn_like(x)

        return x.numpy()


class TabSynLite(TabDDPMLite):
    """
    Compact TabSyn: train a VAE, then run the diffusion in its latent space.

    Latent diffusion is cheaper and usually smoother on mixed-type tables than
    diffusing in raw feature space. Again a re-implementation -- the official
    code is amazon-science/tabsyn.
    """
    name = "TabSyn-lite"

    def __init__(self, cat_cols, seed=SEED, epochs=4000, timesteps=200,
                 hidden=256, lr=1e-3, latent_dim=16, vae_epochs=800):
        super().__init__(cat_cols, seed, epochs, timesteps, hidden, lr)
        self.latent_dim, self.vae_epochs = latent_dim, vae_epochs

    def fit_sample(self, X, y, n_needed):
        import torch
        import torch.nn as nn

        torch.manual_seed(self.seed)
        Z = self._build_transform(X)
        d_in = Z.shape[1]
        Zt = torch.tensor(Z)
        yt = torch.tensor(y.values.astype(np.int64))
        n_classes = int(y.max()) + 1

        # ---- stage 1: conditional VAE ------------------------------------
        class VAE(nn.Module):
            def __init__(self, d_in, h, d_z, n_classes):
                super().__init__()
                self.emb = nn.Embedding(n_classes, 16)
                self.enc = nn.Sequential(nn.Linear(d_in + 16, h), nn.SiLU(),
                                         nn.Linear(h, h), nn.SiLU())
                self.mu, self.logvar = nn.Linear(h, d_z), nn.Linear(h, d_z)
                self.dec = nn.Sequential(nn.Linear(d_z + 16, h), nn.SiLU(),
                                         nn.Linear(h, h), nn.SiLU(),
                                         nn.Linear(h, d_in))

            def encode(self, x, y):
                h = self.enc(torch.cat([x, self.emb(y)], -1))
                return self.mu(h), self.logvar(h).clamp(-8, 8)

            def decode(self, z, y):
                return self.dec(torch.cat([z, self.emb(y)], -1))

        vae = VAE(d_in, self.hidden, self.latent_dim, n_classes)
        opt = torch.optim.AdamW(vae.parameters(), lr=1e-3)
        batch = min(256, len(Zt))
        beta_kl = 1e-3  # small KL weight keeps reconstructions sharp

        for _ in range(self.vae_epochs):
            idx = torch.randint(0, len(Zt), (batch,))
            x0, yb = Zt[idx], yt[idx]
            mu, logvar = vae.encode(x0, yb)
            z = mu + torch.randn_like(mu) * (0.5 * logvar).exp()
            recon = vae.decode(z, yb)
            rec_loss = ((recon - x0) ** 2).mean()
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = rec_loss + beta_kl * kl
            opt.zero_grad(); loss.backward(); opt.step()

        vae.eval()
        with torch.no_grad():
            mu, _ = vae.encode(Zt, yt)
            lat = mu
            lat_mean, lat_std = lat.mean(0, keepdim=True), lat.std(0, keepdim=True) + 1e-6
            lat_n = (lat - lat_mean) / lat_std

        # ---- stage 2: diffusion in latent space --------------------------
        s, T = 0.008, self.timesteps
        steps = np.arange(T + 1) / T
        acum = np.cos((steps + s) / (1 + s) * np.pi / 2) ** 2
        acum /= acum[0]
        betas = np.clip(1 - acum[1:] / acum[:-1], 1e-8, 0.999)
        betas_t = torch.tensor(betas, dtype=torch.float32)
        alphas_t = 1.0 - betas_t
        acp = torch.cumprod(alphas_t, 0)

        net = self._make_net(self.latent_dim, n_classes)
        opt2 = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=1e-5)
        for _ in range(self.epochs):
            idx = torch.randint(0, len(lat_n), (batch,))
            x0, yb = lat_n[idx], yt[idx]
            t = torch.randint(0, T, (batch,))
            noise = torch.randn_like(x0)
            a = acp[t][:, None]
            xt = a.sqrt() * x0 + (1 - a).sqrt() * noise
            loss = ((net(xt, t, yb) - noise) ** 2).mean()
            opt2.zero_grad(); loss.backward(); opt2.step()

        net.eval()
        z = torch.tensor(self._ddpm_sample(
            net, n_needed, self.latent_dim, alphas_t, acp, betas_t, T))
        with torch.no_grad():
            y_gen = torch.ones(n_needed, dtype=torch.long)
            decoded = vae.decode(z * lat_std + lat_mean, y_gen).numpy()

        return self._inverse_transform(decoded)


# =============================================================================
# 3. EVALUATION
# =============================================================================

def classifier_zoo(seed: int = SEED) -> dict:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    zoo = {
        "RandomForest": RandomForestClassifier(
            n_estimators=400, random_state=seed, n_jobs=-1),
        "LogReg": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=3000, random_state=seed)),
    }
    try:
        from xgboost import XGBClassifier
        zoo["XGBoost"] = XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
            random_state=seed, n_jobs=-1, verbosity=0)
    except ImportError:
        pass
    return zoo


def utility_scores(X_tr, y_tr, X_te, y_te, seed=SEED) -> dict:
    """Train on (possibly augmented) train, score on REAL test only."""
    from sklearn.metrics import (accuracy_score, average_precision_score,
                                 f1_score, precision_score, recall_score,
                                 roc_auc_score)

    rows = {}
    for name, clf in classifier_zoo(seed).items():
        clf.fit(X_tr, y_tr)
        pred = clf.predict(X_te)
        prob = (clf.predict_proba(X_te)[:, 1]
                if hasattr(clf, "predict_proba") else pred)
        rows[name] = {
            "Accuracy": accuracy_score(y_te, pred),
            "Precision": precision_score(y_te, pred, zero_division=0),
            "Recall": recall_score(y_te, pred, zero_division=0),
            "F1": f1_score(y_te, pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_te, prob),
            "PR-AUC": average_precision_score(y_te, prob),
        }
    return rows


def fidelity_scores(real: pd.DataFrame, synth: pd.DataFrame,
                    cat_cols: list[str]) -> dict:
    """How closely does synthetic data match the real marginals / correlations?"""
    from scipy.stats import ks_2samp

    if synth.empty:
        return {"KS_mean": np.nan, "TVD_mean": np.nan, "CorrDiff": np.nan}

    common = [c for c in real.columns if c in synth.columns]
    num_cols = [c for c in common if c not in cat_cols]
    cats = [c for c in common if c in cat_cols]

    # Kolmogorov-Smirnov per numeric column (lower = better)
    ks = [ks_2samp(real[c].dropna(), pd.to_numeric(synth[c], errors="coerce").dropna()).statistic
          for c in num_cols if synth[c].notna().any()]

    # Total variation distance per categorical column (lower = better)
    tvd = []
    for c in cats:
        p = real[c].value_counts(normalize=True)
        q = pd.to_numeric(synth[c], errors="coerce").value_counts(normalize=True)
        levels = p.index.union(q.index)
        tvd.append(0.5 * np.abs(p.reindex(levels, fill_value=0)
                                - q.reindex(levels, fill_value=0)).sum())

    # Correlation-matrix difference (lower = better)
    try:
        cd = np.nanmean(np.abs(real[common].corr().values
                               - synth[common].astype(float).corr().values))
    except Exception:
        cd = np.nan

    return {"KS_mean": np.mean(ks) if ks else np.nan,
            "TVD_mean": np.mean(tvd) if tvd else np.nan,
            "CorrDiff": cd}


def privacy_dcr(real: pd.DataFrame, synth: pd.DataFrame) -> float:
    """
    Distance to Closest Record. Median nearest-neighbour distance from each
    synthetic row to the real training data, on standardised features.
    Near 0 means the generator is memorising patients -- a privacy red flag.
    """
    if synth.empty:
        return np.nan
    from sklearn.neighbors import NearestNeighbors
    from sklearn.preprocessing import StandardScaler

    common = [c for c in real.columns if c in synth.columns]
    sc = StandardScaler().fit(real[common])
    nn = NearestNeighbors(n_neighbors=1).fit(sc.transform(real[common]))
    d, _ = nn.kneighbors(sc.transform(synth[common].astype(float)))
    return float(np.median(d))


# =============================================================================
# 4. BENCHMARK DRIVER
# =============================================================================

def run_benchmark(X, y, cat_cols, dataset_name, test_size=0.25,
                  epochs=300, dl_epochs=4000, seed=SEED, skip=()):
    import time
    from sklearn.model_selection import train_test_split

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed)

    n_major = int((y_tr == 0).sum())
    n_minor = int((y_tr == 1).sum())
    n_needed = max(0, n_major - n_minor)

    print(f"\n{'=' * 78}\n  {dataset_name}\n{'=' * 78}")
    print(f"  real train : {X_tr.shape}   class 0={n_major}  class 1={n_minor}")
    print(f"  real test  : {X_te.shape}   class 0={(y_te == 0).sum()}  "
          f"class 1={(y_te == 1).sum()}")
    print(f"  synthetic minority rows to generate: {n_needed}\n")

    generators = [
        ImblearnGenerator("SMOTE", cat_cols, seed),
        ImblearnGenerator("ADASYN", cat_cols, seed),
        ImblearnGenerator("BorderlineSMOTE", cat_cols, seed),
        CTGANGenerator(cat_cols, seed, epochs=epochs),
        TVAEGenerator(cat_cols, seed, epochs=epochs),
        CTABGANPlusGenerator(cat_cols, seed, epochs=epochs),
        TabDDPMLite(cat_cols, seed, epochs=dl_epochs),
        TabSynLite(cat_cols, seed, epochs=dl_epochs),
    ]

    records, fidelity, real_minority = [], [], X_tr[y_tr == 1]

    # -- Baseline: no augmentation at all --------------------------------------
    for clf_name, m in utility_scores(X_tr, y_tr, X_te, y_te, seed).items():
        records.append({"Generator": "None (baseline)", "Classifier": clf_name,
                        "SynthRows": 0, "Seconds": 0.0, **m})

    # -- Each generator --------------------------------------------------------
    for gen in generators:
        if gen.name in skip:
            continue
        label = gen.name
        try:
            t0 = time.time()
            synth = gen.fit_sample(X_tr.copy(), y_tr.copy(), n_needed)
            elapsed = time.time() - t0
            label = gen.name  # CTAB-GAN+ may have renamed itself on fallback

            synth = synth.reindex(columns=X_tr.columns)
            synth = synth.apply(pd.to_numeric, errors="coerce")
            synth = synth.fillna(X_tr.median(numeric_only=True)).dropna()

            X_aug = pd.concat([X_tr, synth], ignore_index=True)
            y_aug = pd.concat([y_tr, pd.Series([1] * len(synth))], ignore_index=True)

            print(f"  [{label:<38}] {len(synth):>4} rows  {elapsed:>6.1f}s")

            for clf_name, m in utility_scores(X_aug, y_aug, X_te, y_te, seed).items():
                records.append({"Generator": label, "Classifier": clf_name,
                                "SynthRows": len(synth),
                                "Seconds": round(elapsed, 1), **m})

            fidelity.append({"Generator": label,
                             **fidelity_scores(real_minority, synth, cat_cols),
                             "DCR_median": privacy_dcr(X_tr, synth)})
        except Exception as exc:
            print(f"  [{label:<38}] FAILED: {type(exc).__name__}: {exc}")

    return pd.DataFrame(records), pd.DataFrame(fidelity)


def print_report(utility: pd.DataFrame, fidelity: pd.DataFrame, tag: str):
    pd.set_option("display.width", 200, "display.max_columns", 50)

    print(f"\n--- UTILITY on real held-out test set ({tag}) ---")
    piv = utility.pivot_table(index="Generator", columns="Classifier",
                              values=["F1", "ROC-AUC"]).round(3)
    print(piv.to_string())

    print(f"\n--- MEAN ACROSS CLASSIFIERS ({tag}) ---")
    mean = (utility.groupby("Generator")[
        ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]]
        .mean().round(3).sort_values("F1", ascending=False))
    print(mean.to_string())

    if not fidelity.empty:
        print(f"\n--- FIDELITY + PRIVACY ({tag}) ---")
        print("  KS/TVD/CorrDiff: lower is better.  DCR: higher = less memorisation.")
        print(fidelity.set_index("Generator").round(3).to_string())


# =============================================================================
# 5. MAIN
# =============================================================================

def main():
    ap = argparse.ArgumentParser(description="Generative AI benchmark on PCOS data")
    ap.add_argument("--clinical", default="/content/drive/MyDrive/PCOD/Training data.csv",
                    help="path to the 541x45 clinical PCOS csv")
    ap.add_argument("--survey", default="/content/drive/MyDrive/PCOD/Testing data.csv",
                    help="path to the 173x36 lifestyle/survey PCOS csv")
    ap.add_argument("--dataset", choices=["clinical", "survey", "both"],
                    default="both")
    ap.add_argument("--epochs", type=int, default=300, help="CTGAN/TVAE epochs")
    ap.add_argument("--dl-epochs", type=int, default=4000,
                    help="diffusion training steps")
    ap.add_argument("--out-prefix", default="pcos_results")

    # parse_known_args (not parse_args) so this survives being run inside a
    # Jupyter/Colab cell, where the kernel injects its own "-f kernel.json"
    args, _unknown = ap.parse_known_args()

    if args.dataset in ("clinical", "both"):
        X, y, cats = load_clinical(args.clinical)
        u, f = run_benchmark(X, y, cats, "DATASET A - clinical PCOS",
                             epochs=args.epochs, dl_epochs=args.dl_epochs)
        print_report(u, f, "clinical")
        u.to_csv(f"{args.out_prefix}_clinical_utility.csv", index=False)
        f.to_csv(f"{args.out_prefix}_clinical_fidelity.csv", index=False)

    if args.dataset in ("survey", "both"):
        X, y, cats = load_survey(args.survey)
        u, f = run_benchmark(X, y, cats, "DATASET B - lifestyle/survey PCOS",
                             epochs=args.epochs, dl_epochs=args.dl_epochs)
        print_report(u, f, "survey")
        u.to_csv(f"{args.out_prefix}_survey_utility.csv", index=False)
        f.to_csv(f"{args.out_prefix}_survey_fidelity.csv", index=False)

    print("\nDone. CSV result files written with prefix "
          f"'{args.out_prefix}_'.")


if __name__ == "__main__":
    main()


  DATASET A - clinical PCOS
  real train : (405, 41)   class 0=272  class 1=133
  real test  : (136, 41)   class 0=92  class 1=44
  synthetic minority rows to generate: 139

  [SMOTE                                 ]  139 rows     0.8s
  [ADASYN                                ]  139 rows     0.0s
  [BorderlineSMOTE                       ]  139 rows     0.0s
  [CTGAN                                 ]  139 rows    57.1s
  [TVAE                                  ]  139 rows    15.8s
  [CTAB-GAN+ (unavailable -> CTGAN fallback)]  139 rows    48.3s
  [TabDDPM-lite                          ]  139 rows    40.5s
  [TabSyn-lite                           ]  139 rows    46.1s

--- UTILITY on real held-out test set (clinical) ---
                                              F1                      ROC-AUC                     
Classifier                                LogReg RandomForest XGBoost  LogReg RandomForest XGBoost
Generator                                                                 

## generative AI - with visulization

In [4]:
"""
===============================================================================
  GENERATIVE AI FOR TABULAR PCOS DATA
  Benchmark + Visualisation + Final Model Verdict
===============================================================================

  Generators compared
  -------------------
    Classical (interpolation) : SMOTE, ADASYN, BorderlineSMOTE
    GAN                       : CTGAN, CTAB-GAN+ (adapter)
    VAE                       : TVAE
    Diffusion                 : TabDDPM-lite   (compact re-implementation)
    Latent diffusion          : TabSyn-lite    (compact re-implementation)

  Protocol
  --------
    for each repeat r:
        1. stratified split of REAL data      -> real_train / real_test
        2. generator fitted on real_train ONLY
        3. synthetic minority rows appended to real_train
        4. classifiers trained on the augmented set
        5. scored on real_test  -- NEVER on synthetic rows
    metrics averaged over repeats, reported as mean +/- std

  Outputs
  -------
    *_utility.csv / *_fidelity.csv     raw per-run metrics
    plots/01 .. 10 *.png               ten visualisations
    printed FINAL VERDICT              composite-ranked winner

  Install
  -------
    pip install pandas numpy scipy scikit-learn imbalanced-learn ctgan torch
    pip install matplotlib seaborn
    pip install xgboost          # optional third classifier
===============================================================================
"""

from __future__ import annotations

import argparse
import os
import time
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# Consistent colour per generator family across every chart
FAMILY_COLOURS = {
    "None (baseline)": "#9e9e9e",
    "SMOTE": "#4C9F70", "ADASYN": "#6FBF8C", "BorderlineSMOTE": "#2E7D50",
    "CTGAN": "#D96C4F", "CTAB-GAN+": "#B04A2E",
    "TVAE": "#E0A458",
    "TabDDPM-lite": "#4A6FA5", "TabSyn-lite": "#7189BF",
}


def colour_for(name: str) -> str:
    for key, col in FAMILY_COLOURS.items():
        if name.startswith(key):
            return col
    return "#888888"


# =============================================================================
#  SECTION 1 — DATA LOADING AND CLEANING
# =============================================================================

def load_clinical(path: str):
    """Clinical PCOS file (541 x 45). Returns (X, y, categorical_columns)."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    for junk in ["Sl. No", "Patient File No.", "Unnamed: 44"]:
        if junk in df.columns:
            df = df.drop(columns=junk)
    df = df.dropna(axis=1, how="all")

    target = "PCOS (Y/N)"
    y = pd.to_numeric(df[target], errors="coerce")
    X = df.drop(columns=target)

    # Two columns carry dirty strings ("1.99.", "a") -> coerce to NaN
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    cat_cols = [c for c in X.columns if X[c].dropna().nunique() <= 2]
    if "Blood Group" in X.columns:
        cat_cols.append("Blood Group")

    keep = y.notna()
    X, y = X.loc[keep], y.loc[keep].astype(int)
    X = X.fillna(X.median(numeric_only=True))
    return X.reset_index(drop=True), y.reset_index(drop=True), cat_cols


def load_survey(path: str):
    """Lifestyle/survey PCOS file (173 x 36). Returns (X, y, categorical_columns)."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    # Free-text answers -- unusable as a numeric feature
    df = df.drop(columns=[c for c in ["PCOS_Medication"] if c in df.columns])

    df["PCOS"] = df["PCOS"].astype(str).str.strip()
    df = df[df["PCOS"].isin(["Yes", "No"])].copy()   # drops 1 ambiguous row
    y = (df["PCOS"] == "Yes").astype(int)
    X = df.drop(columns="PCOS")

    # Ordinal encodings preserve real ordering (better than one-hot here)
    ordinal_maps = {
        "Age": {"Less than 20": 0, "20-25": 1, "25-30": 2,
                "30-35": 3, "35-44": 4, "45 and above": 5},
        "Exercise_Frequency": {"Never": 0, "Rarely": 1, "1-2 Times a Week": 2,
                               "3-4 Times a Week": 3, "Daily": 4},
        "Exercise_Duration": {"Not Applicable": 0, "Less than 30 minutes": 1,
                              "30 minutes": 2, "More than 30 minutes": 3,
                              "30 minutes to 1 hour": 4},
        "Sleep_Hours": {"Less than 6 hours": 0, "6-8 hours": 1,
                        "9-12 hours": 2, "More than 12 hours": 3},
        "Exercise_Benefit": {"Not at All": 0, "Not Much": 1,
                             "Somewhat": 2, "Yes Significantly": 3},
    }
    for col, mapping in ordinal_maps.items():
        if col in X.columns:
            X[col] = X[col].astype(str).str.strip().map(mapping)

    cat_cols = []
    for col in X.columns:
        if X[col].dtype == object or str(X[col].dtype) == "str":
            X[col] = pd.factorize(X[col].astype(str).str.strip())[0]
            cat_cols.append(col)
        elif X[col].dropna().nunique() <= 2:
            cat_cols.append(col)

    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    return X.reset_index(drop=True), y.reset_index(drop=True), cat_cols


# =============================================================================
#  SECTION 2 — GENERATORS
# =============================================================================

class BaseGenerator:
    """Interface: fit_sample(X_train, y_train, n_needed) -> DataFrame."""
    name = "base"

    def __init__(self, cat_cols, seed=SEED):
        self.cat_cols, self.seed = cat_cols, seed

    def fit_sample(self, X, y, n_needed):
        raise NotImplementedError


# ---------------------------------------------------------------- classical --
class ImblearnGenerator(BaseGenerator):
    """SMOTE family. Interpolates between neighbours; learns no density."""

    def __init__(self, kind, cat_cols, seed=SEED):
        super().__init__(cat_cols, seed)
        self.kind = self.name = kind

    def fit_sample(self, X, y, n_needed):
        from imblearn.over_sampling import ADASYN, SMOTE, BorderlineSMOTE

        k = int(min(5, max(1, (y == 1).sum() - 1)))
        algo = {"SMOTE": SMOTE(random_state=self.seed, k_neighbors=k),
                "ADASYN": ADASYN(random_state=self.seed, n_neighbors=k),
                "BorderlineSMOTE": BorderlineSMOTE(random_state=self.seed,
                                                   k_neighbors=k)}[self.kind]
        Xr, _ = algo.fit_resample(X, y)
        synth = Xr.iloc[len(X):].copy()            # appended rows are synthetic
        for c in self.cat_cols:
            if c in synth.columns:
                synth[c] = synth[c].round().astype(int)
        return synth.head(n_needed)


# ---------------------------------------------------------------- GAN / VAE --
class CTGANGenerator(BaseGenerator):
    """Conditional Tabular GAN (Xu et al., 2019)."""
    name = "CTGAN"

    def __init__(self, cat_cols, seed=SEED, epochs=300, cuda=False):
        super().__init__(cat_cols, seed)
        self.epochs, self.cuda = epochs, cuda

    def fit_sample(self, X, y, n_needed):
        from ctgan import CTGAN
        return self._run(CTGAN, X, y, n_needed, verbose_kw=True)

    def _run(self, model_cls, X, y, n_needed, verbose_kw=False):
        train = X.copy()
        train["__label__"] = y.values
        discrete = [c for c in self.cat_cols if c in train.columns] + ["__label__"]

        kwargs = dict(epochs=self.epochs, cuda=self.cuda)
        if verbose_kw:
            kwargs["verbose"] = False

        self._collapsed = False
        model = model_cls(**kwargs)
        model.fit(train, discrete_columns=discrete)
        synth = self._sample_minority(model, n_needed)

        if self._collapsed:
            # GANs/VAEs mode-collapse on small imbalanced tables and stop
            # emitting the minority label. Refit on the minority subset instead
            # of silently returning nothing.
            self.name += " [minority-only fallback]"
            minority = X[y == 1].copy()
            m2 = model_cls(**kwargs)
            m2.fit(minority, discrete_columns=[c for c in self.cat_cols
                                               if c in minority.columns])
            synth = m2.sample(n_needed)
        return synth

    def _sample_minority(self, model, n_needed):
        pool, total, tries = [], 0, 0
        while total < n_needed and tries < 12:
            batch = model.sample(max(500, n_needed * 6))
            lab = pd.to_numeric(batch["__label__"], errors="coerce").round()
            keep = batch[lab == 1]
            if len(keep):
                pool.append(keep)
                total += len(keep)
            tries += 1
        if total == 0:
            self._collapsed = True
            return pd.DataFrame()
        return pd.concat(pool, ignore_index=True).drop(
            columns="__label__", errors="ignore").head(n_needed)


class TVAEGenerator(CTGANGenerator):
    """Tabular VAE -- same paper as CTGAN, variational instead of adversarial."""
    name = "TVAE"

    def fit_sample(self, X, y, n_needed):
        from ctgan import TVAE
        return self._run(TVAE, X, y, n_needed, verbose_kw=False)


class CTABGANPlusGenerator(CTGANGenerator):
    """
    CTAB-GAN+ adapter.

    Not on PyPI -- it is a research repo:
        git clone https://github.com/Team-TUD/CTAB-GAN-Plus
    Set CTABGAN_REPO to the clone path. Without it, this falls back to CTGAN
    and RENAMES ITSELF in every output, so a fallback number can never be
    mistaken for a real CTAB-GAN+ result.
    """
    name = "CTAB-GAN+"
    CTABGAN_REPO = None

    def fit_sample(self, X, y, n_needed):
        if self.CTABGAN_REPO is None:
            self.name = "CTAB-GAN+ (unavailable -> CTGAN)"
            from ctgan import CTGAN
            return self._run(CTGAN, X, y, n_needed, verbose_kw=True)

        import sys
        sys.path.insert(0, self.CTABGAN_REPO)
        from model.ctabgan import CTABGAN            # type: ignore

        train = X.copy()
        train["__label__"] = y.values
        tmp = "_ctabgan_input.csv"
        train.to_csv(tmp, index=False)
        model = CTABGAN(raw_csv_path=tmp,
                        categorical_columns=[c for c in self.cat_cols
                                             if c in train.columns],
                        problem_type={"Classification": "__label__"},
                        epochs=self.epochs)
        model.fit()
        pool = model.generate_samples(max(500, n_needed * 6))
        return pool[pool["__label__"] == 1].drop(
            columns="__label__").head(n_needed)


# ------------------------------------------------------------- diffusion -----
class TabDDPMLite(BaseGenerator):
    """
    Compact class-conditional Gaussian diffusion for tabular data.

    Follows the TabDDPM recipe (Kotelnikov et al., 2023) in spirit:
      numeric      -> quantile-normalised to ~N(0,1)
      categorical  -> one-hot, diffused in the same Gaussian space
      denoiser     -> MLP conditioned on a timestep embedding + class label
      sampling     -> ancestral DDPM with the label pinned to the minority class

    Re-implementation, NOT the official repo (yandex-research/tab-ddpm).
    State that clearly in any write-up.
    """
    name = "TabDDPM-lite"

    def __init__(self, cat_cols, seed=SEED, epochs=4000, timesteps=200,
                 hidden=256, lr=1e-3):
        super().__init__(cat_cols, seed)
        self.epochs, self.timesteps = epochs, timesteps
        self.hidden, self.lr = hidden, lr

    # -- preprocessing -------------------------------------------------------
    def _build_transform(self, X):
        from sklearn.preprocessing import OneHotEncoder, QuantileTransformer

        self.num_cols = [c for c in X.columns if c not in self.cat_cols]
        self.cat_cols_ = [c for c in self.cat_cols if c in X.columns]

        self.qt = QuantileTransformer(
            output_distribution="normal",
            n_quantiles=max(10, min(1000, len(X) // 2)),
            random_state=self.seed)
        num = (self.qt.fit_transform(X[self.num_cols])
               if self.num_cols else np.zeros((len(X), 0)))

        if self.cat_cols_:
            self.ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            cat = self.ohe.fit_transform(X[self.cat_cols_].astype(str))
            self.cat_sizes = [len(c) for c in self.ohe.categories_]
        else:
            self.ohe, cat, self.cat_sizes = None, np.zeros((len(X), 0)), []

        return np.hstack([num, cat]).astype(np.float32)

    def _inverse_transform(self, Z):
        n_num = len(self.num_cols)
        out = {}
        if n_num:
            num = self.qt.inverse_transform(Z[:, :n_num])
            for i, c in enumerate(self.num_cols):
                out[c] = num[:, i]

        offset = n_num
        cats_list = self.ohe.categories_ if self.ohe is not None else []
        for size, col, cats in zip(self.cat_sizes, self.cat_cols_, cats_list):
            out[col] = np.asarray(cats)[Z[:, offset:offset + size].argmax(1)]
            offset += size

        df = pd.DataFrame(out)
        for c in self.cat_cols_:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        return df

    # -- denoiser ------------------------------------------------------------
    def _make_net(self, d_in, n_classes):
        import torch
        import torch.nn as nn

        class Denoiser(nn.Module):
            def __init__(self, d_in, hidden, n_classes, t_dim=64):
                super().__init__()
                self.t_dim = t_dim
                self.label_emb = nn.Embedding(n_classes, t_dim)
                self.net = nn.Sequential(
                    nn.Linear(d_in + t_dim * 2, hidden), nn.SiLU(),
                    nn.Linear(hidden, hidden), nn.SiLU(),
                    nn.Linear(hidden, hidden), nn.SiLU(),
                    nn.Linear(hidden, d_in))

            def timestep_embedding(self, t):
                half = self.t_dim // 2
                freqs = torch.exp(-np.log(10000)
                                  * torch.arange(half, device=t.device) / half)
                args = t.float()[:, None] * freqs[None]
                return torch.cat([args.cos(), args.sin()], -1)

            def forward(self, x, t, y):
                h = torch.cat([x, self.timestep_embedding(t),
                               self.label_emb(y)], -1)
                return self.net(h)

        return Denoiser(d_in, self.hidden, n_classes)

    @staticmethod
    def _cosine_schedule(T):
        """Nichol & Dhariwal cosine schedule -- more stable than linear."""
        import torch
        s = 0.008
        steps = np.arange(T + 1) / T
        acum = np.cos((steps + s) / (1 + s) * np.pi / 2) ** 2
        acum /= acum[0]
        betas = np.clip(1 - acum[1:] / acum[:-1], 1e-8, 0.999)
        betas_t = torch.tensor(betas, dtype=torch.float32)
        alphas_t = 1.0 - betas_t
        return betas_t, alphas_t, torch.cumprod(alphas_t, 0)

    @staticmethod
    def _ddpm_sample(net, n, d, alphas_t, acp, betas_t, T, clamp=4.0):
        """
        Ancestral sampling with CLAMPED x0 prediction + true posterior variance.

        Without the clamp, small tabular datasets produce synthetic rows with
        2-3x the real standard deviation and the KS fidelity score collapses.
        """
        import torch
        with torch.no_grad():
            x = torch.randn(n, d)
            y_gen = torch.ones(n, dtype=torch.long)
            acp_prev = torch.cat([torch.ones(1), acp[:-1]])
            for t in reversed(range(T)):
                tt = torch.full((n,), t, dtype=torch.long)
                eps = net(x, tt, y_gen)
                x0 = ((x - (1 - acp[t]).sqrt() * eps)
                      / acp[t].sqrt()).clamp(-clamp, clamp)
                mean = (acp_prev[t].sqrt() * betas_t[t] / (1 - acp[t]) * x0
                        + alphas_t[t].sqrt() * (1 - acp_prev[t])
                        / (1 - acp[t]) * x)
                if t == 0:
                    x = mean
                else:
                    var = betas_t[t] * (1 - acp_prev[t]) / (1 - acp[t])
                    x = mean + var.sqrt() * torch.randn_like(x)
        return x.numpy()

    def fit_sample(self, X, y, n_needed):
        import torch

        torch.manual_seed(self.seed)
        Z = self._build_transform(X)
        y_arr = y.values.astype(np.int64)
        T = self.timesteps
        betas_t, alphas_t, acp = self._cosine_schedule(T)

        Zt, yt = torch.tensor(Z), torch.tensor(y_arr)
        net = self._make_net(Z.shape[1], int(y_arr.max()) + 1)
        opt = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=1e-5)
        batch = min(256, len(Zt))

        for _ in range(self.epochs):
            idx = torch.randint(0, len(Zt), (batch,))
            x0, yb = Zt[idx], yt[idx]
            t = torch.randint(0, T, (batch,))
            noise = torch.randn_like(x0)
            a = acp[t][:, None]
            xt = a.sqrt() * x0 + (1 - a).sqrt() * noise
            loss = ((net(xt, t, yb) - noise) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()

        net.eval()
        x = self._ddpm_sample(net, n_needed, Z.shape[1],
                              alphas_t, acp, betas_t, T)
        return self._inverse_transform(x)


class TabSynLite(TabDDPMLite):
    """
    Compact TabSyn: VAE first, then diffusion inside its latent space.

    Latent diffusion is cheaper and usually smoother on mixed-type tables.
    Re-implementation -- official code is amazon-science/tabsyn.
    """
    name = "TabSyn-lite"

    def __init__(self, cat_cols, seed=SEED, epochs=4000, timesteps=200,
                 hidden=256, lr=1e-3, latent_dim=16, vae_epochs=1500):
        super().__init__(cat_cols, seed, epochs, timesteps, hidden, lr)
        self.latent_dim, self.vae_epochs = latent_dim, vae_epochs

    def fit_sample(self, X, y, n_needed):
        import torch
        import torch.nn as nn

        torch.manual_seed(self.seed)
        Z = self._build_transform(X)
        d_in = Z.shape[1]
        Zt = torch.tensor(Z)
        yt = torch.tensor(y.values.astype(np.int64))
        n_classes = int(y.max()) + 1

        # ---- stage 1: conditional VAE -------------------------------------
        class VAE(nn.Module):
            def __init__(self, d_in, h, d_z, n_classes):
                super().__init__()
                self.emb = nn.Embedding(n_classes, 16)
                self.enc = nn.Sequential(nn.Linear(d_in + 16, h), nn.SiLU(),
                                         nn.Linear(h, h), nn.SiLU())
                self.mu = nn.Linear(h, d_z)
                self.logvar = nn.Linear(h, d_z)
                self.dec = nn.Sequential(nn.Linear(d_z + 16, h), nn.SiLU(),
                                         nn.Linear(h, h), nn.SiLU(),
                                         nn.Linear(h, d_in))

            def encode(self, x, y):
                h = self.enc(torch.cat([x, self.emb(y)], -1))
                return self.mu(h), self.logvar(h).clamp(-8, 8)

            def decode(self, z, y):
                return self.dec(torch.cat([z, self.emb(y)], -1))

        vae = VAE(d_in, self.hidden, self.latent_dim, n_classes)
        opt = torch.optim.AdamW(vae.parameters(), lr=1e-3)
        batch = min(256, len(Zt))
        beta_kl = 1e-3            # small KL weight keeps reconstructions sharp

        for _ in range(self.vae_epochs):
            idx = torch.randint(0, len(Zt), (batch,))
            x0, yb = Zt[idx], yt[idx]
            mu, logvar = vae.encode(x0, yb)
            z = mu + torch.randn_like(mu) * (0.5 * logvar).exp()
            rec = ((vae.decode(z, yb) - x0) ** 2).mean()
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = rec + beta_kl * kl
            opt.zero_grad(); loss.backward(); opt.step()

        vae.eval()
        with torch.no_grad():
            lat, _ = vae.encode(Zt, yt)
            lat_mean = lat.mean(0, keepdim=True)
            lat_std = lat.std(0, keepdim=True) + 1e-6
            lat_n = (lat - lat_mean) / lat_std

        # ---- stage 2: diffusion in latent space ---------------------------
        T = self.timesteps
        betas_t, alphas_t, acp = self._cosine_schedule(T)
        net = self._make_net(self.latent_dim, n_classes)
        opt2 = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=1e-5)

        for _ in range(self.epochs):
            idx = torch.randint(0, len(lat_n), (batch,))
            x0, yb = lat_n[idx], yt[idx]
            t = torch.randint(0, T, (batch,))
            noise = torch.randn_like(x0)
            a = acp[t][:, None]
            xt = a.sqrt() * x0 + (1 - a).sqrt() * noise
            loss = ((net(xt, t, yb) - noise) ** 2).mean()
            opt2.zero_grad(); loss.backward(); opt2.step()

        net.eval()
        z = torch.tensor(self._ddpm_sample(net, n_needed, self.latent_dim,
                                           alphas_t, acp, betas_t, T))
        with torch.no_grad():
            y_gen = torch.ones(n_needed, dtype=torch.long)
            decoded = vae.decode(z * lat_std + lat_mean, y_gen).numpy()
        return self._inverse_transform(decoded)


# =============================================================================
#  SECTION 3 — EVALUATION METRICS
# =============================================================================

def classifier_zoo(seed=SEED):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    zoo = {
        "RandomForest": RandomForestClassifier(n_estimators=400,
                                               random_state=seed, n_jobs=-1),
        "LogReg": make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=3000,
                                                   random_state=seed)),
    }
    try:
        from xgboost import XGBClassifier
        zoo["XGBoost"] = XGBClassifier(n_estimators=400, max_depth=4,
                                       learning_rate=0.08, subsample=0.9,
                                       colsample_bytree=0.9,
                                       eval_metric="logloss", random_state=seed,
                                       n_jobs=-1, verbosity=0)
    except ImportError:
        pass
    return zoo


def utility_scores(X_tr, y_tr, X_te, y_te, seed=SEED, keep_curves=False):
    """Train on the augmented set, score on the REAL test set only."""
    from sklearn.metrics import (accuracy_score, average_precision_score,
                                 f1_score, precision_score, recall_score,
                                 roc_auc_score, roc_curve)

    rows, curves = {}, {}
    for name, clf in classifier_zoo(seed).items():
        clf.fit(X_tr, y_tr)
        pred = clf.predict(X_te)
        prob = (clf.predict_proba(X_te)[:, 1]
                if hasattr(clf, "predict_proba") else pred)
        rows[name] = {"Accuracy": accuracy_score(y_te, pred),
                      "Precision": precision_score(y_te, pred, zero_division=0),
                      "Recall": recall_score(y_te, pred, zero_division=0),
                      "F1": f1_score(y_te, pred, zero_division=0),
                      "ROC-AUC": roc_auc_score(y_te, prob),
                      "PR-AUC": average_precision_score(y_te, prob)}
        if keep_curves:
            fpr, tpr, _ = roc_curve(y_te, prob)
            curves[name] = (fpr, tpr, rows[name]["ROC-AUC"])
    return rows, curves          # curves is empty unless keep_curves


def fidelity_scores(real, synth, cat_cols):
    """Marginal + correlation agreement between real and synthetic minority."""
    from scipy.stats import ks_2samp

    if synth is None or synth.empty:
        return {"KS_mean": np.nan, "TVD_mean": np.nan, "CorrDiff": np.nan}

    common = [c for c in real.columns if c in synth.columns]
    num_cols = [c for c in common if c not in cat_cols]
    cats = [c for c in common if c in cat_cols]

    ks = [ks_2samp(real[c].dropna(),
                   pd.to_numeric(synth[c], errors="coerce").dropna()).statistic
          for c in num_cols if synth[c].notna().any()]

    tvd = []
    for c in cats:
        p = real[c].value_counts(normalize=True)
        q = pd.to_numeric(synth[c], errors="coerce").value_counts(normalize=True)
        lv = p.index.union(q.index)
        tvd.append(0.5 * np.abs(p.reindex(lv, fill_value=0)
                                - q.reindex(lv, fill_value=0)).sum())

    try:
        cd = np.nanmean(np.abs(real[common].corr().values
                               - synth[common].astype(float).corr().values))
    except Exception:
        cd = np.nan

    return {"KS_mean": np.mean(ks) if ks else np.nan,
            "TVD_mean": np.mean(tvd) if tvd else np.nan,
            "CorrDiff": cd}


def privacy_dcr(real, synth):
    """
    Distance to Closest Record: median nearest-neighbour distance from each
    synthetic row to a real training row, on standardised features.
    Near 0 = the generator is copying real patients (privacy red flag).
    """
    if synth is None or synth.empty:
        return np.nan
    from sklearn.neighbors import NearestNeighbors
    from sklearn.preprocessing import StandardScaler

    common = [c for c in real.columns if c in synth.columns]
    sc = StandardScaler().fit(real[common])
    nn = NearestNeighbors(n_neighbors=1).fit(sc.transform(real[common]))
    d, _ = nn.kneighbors(sc.transform(synth[common].astype(float)))
    return float(np.median(d))


# =============================================================================
#  SECTION 4 — BENCHMARK DRIVER (repeated splits)
# =============================================================================

@dataclass
class BenchmarkOutput:
    utility: pd.DataFrame
    fidelity: pd.DataFrame
    samples: dict          # generator -> synthetic DataFrame (first repeat)
    curves: dict           # generator -> {classifier: (fpr, tpr, auc)}
    real_train: pd.DataFrame
    real_minority: pd.DataFrame
    cat_cols: list
    dataset_name: str


def build_generators(cat_cols, seed, epochs, dl_epochs, cuda=False):
    return [
        ImblearnGenerator("SMOTE", cat_cols, seed),
        ImblearnGenerator("ADASYN", cat_cols, seed),
        ImblearnGenerator("BorderlineSMOTE", cat_cols, seed),
        CTGANGenerator(cat_cols, seed, epochs=epochs, cuda=cuda),
        TVAEGenerator(cat_cols, seed, epochs=epochs, cuda=cuda),
        CTABGANPlusGenerator(cat_cols, seed, epochs=epochs, cuda=cuda),
        TabDDPMLite(cat_cols, seed, epochs=dl_epochs),
        TabSynLite(cat_cols, seed, epochs=dl_epochs),
    ]


def run_benchmark(X, y, cat_cols, dataset_name, test_size=0.25, repeats=3,
                  epochs=300, dl_epochs=4000, base_seed=SEED, cuda=False):
    from sklearn.model_selection import train_test_split

    print(f"\n{'=' * 78}\n  {dataset_name}   ({repeats} repeated splits)\n{'=' * 78}")

    util_rows, fid_rows = [], []
    samples, curves = {}, {}
    real_train = real_minority = None

    for rep in range(repeats):
        seed = base_seed + rep
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=seed)

        n_needed = max(0, int((y_tr == 0).sum()) - int((y_tr == 1).sum()))
        if rep == 0:
            real_train, real_minority = X_tr.copy(), X_tr[y_tr == 1].copy()
            print(f"  train {X_tr.shape}  test {X_te.shape}  "
                  f"minority rows to generate: {n_needed}")

        # ---- baseline, no augmentation ------------------------------------
        base_rows, base_curves = utility_scores(X_tr, y_tr, X_te, y_te, seed,
                                                keep_curves=(rep == 0))
        for clf, m in base_rows.items():
            util_rows.append({"Run": rep, "Generator": "None (baseline)",
                              "Classifier": clf, "Seconds": 0.0, **m})
        if rep == 0:
            curves["None (baseline)"] = base_curves

        # ---- each generator ------------------------------------------------
        for gen in build_generators(cat_cols, seed, epochs, dl_epochs, cuda):
            label = gen.name
            try:
                t0 = time.time()
                synth = gen.fit_sample(X_tr.copy(), y_tr.copy(), n_needed)
                elapsed = time.time() - t0
                label = gen.name       # may have been renamed by a fallback

                synth = (synth.reindex(columns=X_tr.columns)
                              .apply(pd.to_numeric, errors="coerce")
                              .fillna(X_tr.median(numeric_only=True))
                              .dropna())

                X_aug = pd.concat([X_tr, synth], ignore_index=True)
                y_aug = pd.concat([y_tr, pd.Series([1] * len(synth))],
                                  ignore_index=True)

                rows, cur = utility_scores(X_aug, y_aug, X_te, y_te, seed,
                                                   keep_curves=(rep == 0))
                for clf, m in rows.items():
                    util_rows.append({"Run": rep, "Generator": label,
                                      "Classifier": clf,
                                      "Seconds": round(elapsed, 1), **m})

                fid_rows.append({"Run": rep, "Generator": label,
                                 **fidelity_scores(X_tr[y_tr == 1], synth, cat_cols),
                                 "DCR_median": privacy_dcr(X_tr, synth)})

                if rep == 0:
                    samples[label], curves[label] = synth, cur
                print(f"    run {rep}  [{label:<34}] {len(synth):>4} rows "
                      f"{elapsed:>6.1f}s")
            except Exception as exc:
                print(f"    run {rep}  [{label:<34}] FAILED: "
                      f"{type(exc).__name__}: {exc}")

    return BenchmarkOutput(pd.DataFrame(util_rows), pd.DataFrame(fid_rows),
                           samples, curves, real_train, real_minority,
                           cat_cols, dataset_name)


# =============================================================================
#  SECTION 5 — VISUALISATION  (ten charts)
# =============================================================================

def _setup_plotting():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight",
        "font.size": 9, "axes.titlesize": 11, "axes.titleweight": "bold",
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": ":",
    })
    return plt


def make_all_plots(out: BenchmarkOutput, ranking: pd.DataFrame, outdir: str):
    plt = _setup_plotting()
    os.makedirs(outdir, exist_ok=True)
    tag = out.dataset_name.replace(" ", "_").replace("/", "-")
    saved = []

    def save(fig, n, name):
        path = os.path.join(outdir, f"{n:02d}_{name}_{tag}.png")
        fig.savefig(path)
        plt.close(fig)
        saved.append(path)

    num_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC",
                "PR-AUC", "Seconds"]
    agg = out.utility.groupby("Generator")[num_cols].agg(["mean", "std"])
    gens = list(agg.index)
    cols = [colour_for(g) for g in gens]

    # ---- 1. utility bars with error bars ----------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, metric in zip(axes, ["F1", "ROC-AUC", "Recall"]):
        m = agg[(metric, "mean")]
        s = agg[(metric, "std")].fillna(0)
        order = m.sort_values(ascending=False).index
        ax.bar(range(len(order)), m[order], yerr=s[order], capsize=3,
               color=[colour_for(g) for g in order], edgecolor="white")
        base = m.get("None (baseline)", np.nan)
        ax.axhline(base, ls="--", lw=1.2, color="crimson", zorder=3,
                   label="no augmentation")
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(order, rotation=55, ha="right", fontsize=7)
        ax.set_title(f"{metric} on real held-out test")
        ax.set_ylim(max(0, m.min() - 0.15), min(1.0, m.max() + 0.08))
        ax.legend(fontsize=7)
    fig.suptitle(f"Downstream utility — {out.dataset_name}", y=1.03,
                 fontweight="bold")
    save(fig, 1, "utility_bars")

    # ---- 2. metric heatmap -------------------------------------------------
    metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
    mat = out.utility.groupby("Generator")[metrics].mean()
    mat = mat.loc[mat["F1"].sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(mat) + 2))
    im = ax.imshow(mat.values, cmap="YlGnBu", aspect="auto")
    ax.set_xticks(range(len(metrics)), metrics, rotation=35, ha="right")
    ax.set_yticks(range(len(mat)), mat.index, fontsize=8)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7,
                    color="white" if v > mat.values.mean() else "black")
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.7, label="score")
    ax.set_title(f"All metrics × all generators — {out.dataset_name}")
    save(fig, 2, "metric_heatmap")

    # ---- 3. radar chart (top 5 by F1) -------------------------------------
    top5 = mat.head(5)
    labels = metrics
    ang = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    ang += ang[:1]
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    for gname, row in top5.iterrows():
        vals = row[labels].tolist() + [row[labels[0]]]
        ax.plot(ang, vals, lw=2, label=gname, color=colour_for(gname))
        ax.fill(ang, vals, alpha=0.07, color=colour_for(gname))
    ax.set_xticks(ang[:-1], labels, fontsize=8)
    ax.set_ylim(max(0, top5.values.min() - 0.1), 1.0)
    ax.set_title(f"Multi-metric profile, top 5 — {out.dataset_name}", pad=22)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.12), fontsize=7)
    save(fig, 3, "radar")

    # ---- 4. PCA: real minority vs synthetic --------------------------------
    if out.samples:
        from sklearn.decomposition import PCA
        from sklearn.preprocessing import StandardScaler

        sc = StandardScaler().fit(out.real_train)
        pca = PCA(n_components=2, random_state=SEED).fit(sc.transform(out.real_train))
        real_p = pca.transform(sc.transform(out.real_minority))

        names = list(out.samples)
        ncol = 4
        nrow = int(np.ceil(len(names) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.2 * nrow))
        for ax, gname in zip(np.ravel(axes), names):
            syn = out.samples[gname].reindex(columns=out.real_train.columns)
            syn_p = pca.transform(sc.transform(syn.astype(float)))
            ax.scatter(real_p[:, 0], real_p[:, 1], s=16, alpha=0.55,
                       label="real PCOS", color="#333333")
            ax.scatter(syn_p[:, 0], syn_p[:, 1], s=16, alpha=0.55,
                       label="synthetic", color=colour_for(gname))
            ax.set_title(gname, fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
            ax.legend(fontsize=6, loc="best")
        for ax in np.ravel(axes)[len(names):]:
            ax.axis("off")
        fig.suptitle("PCA — do synthetic rows land where real patients live?",
                     y=1.0, fontweight="bold")
        save(fig, 4, "pca_overlap")

    # ---- 5. feature distribution overlays (best generator) -----------------
    best = ranking.index[0]
    if best in out.samples:
        num = [c for c in out.real_minority.columns if c not in out.cat_cols]
        # pick the 6 numeric features with the widest real spread
        pick = (out.real_minority[num].std() / (out.real_minority[num].mean().abs() + 1e-9)
                ).sort_values(ascending=False).head(6).index
        fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
        for ax, col in zip(np.ravel(axes), pick):
            r = out.real_minority[col].dropna()
            s = pd.to_numeric(out.samples[best][col], errors="coerce").dropna()
            bins = np.histogram_bin_edges(np.concatenate([r, s]), bins=18)
            ax.hist(r, bins=bins, alpha=0.6, density=True,
                    label="real", color="#333333")
            ax.hist(s, bins=bins, alpha=0.6, density=True,
                    label=best, color=colour_for(best))
            ax.set_title(col, fontsize=9)
            ax.legend(fontsize=7)
        fig.suptitle(f"Marginal distributions — real vs {best}",
                     y=1.0, fontweight="bold")
        save(fig, 5, "distributions")

    # ---- 6. correlation-structure difference -------------------------------
    if out.samples:
        names = list(out.samples)
        ncol = 4
        nrow = int(np.ceil(len(names) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 3.0 * nrow))
        real_corr = out.real_minority.corr().values
        for ax, gname in zip(np.ravel(axes), names):
            syn = out.samples[gname].reindex(columns=out.real_minority.columns)
            diff = np.abs(real_corr - syn.astype(float).corr().values)
            im = ax.imshow(diff, cmap="magma_r", vmin=0, vmax=0.6)
            ax.set_title(f"{gname}\nmean |Δρ| = {np.nanmean(diff):.3f}",
                         fontsize=8)
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for ax in np.ravel(axes)[len(names):]:
            ax.axis("off")
        fig.colorbar(im, ax=np.ravel(axes).tolist(), shrink=0.55,
                     label="|real ρ − synthetic ρ|")
        fig.suptitle("Correlation structure error (darker = better)",
                     y=1.0, fontweight="bold")
        save(fig, 6, "corr_diff")

    # ---- 7. fidelity + privacy --------------------------------------------
    if not out.fidelity.empty:
        f = out.fidelity.groupby("Generator")[
            ["KS_mean", "TVD_mean", "CorrDiff", "DCR_median"]].mean()
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

        fsort = f.sort_values("KS_mean")
        idx = np.arange(len(fsort))
        w = 0.27
        for k, (mname, c) in enumerate(zip(["KS_mean", "TVD_mean", "CorrDiff"],
                                           ["#4A6FA5", "#D96C4F", "#4C9F70"])):
            axes[0].bar(idx + (k - 1) * w, fsort[mname], w, label=mname, color=c)
        axes[0].set_xticks(idx, fsort.index, rotation=55, ha="right", fontsize=7)
        axes[0].set_title("Fidelity error — lower is better")
        axes[0].legend(fontsize=7)

        dsort = f["DCR_median"].sort_values()
        axes[1].barh(range(len(dsort)), dsort.values,
                     color=[colour_for(g) for g in dsort.index])
        axes[1].axvline(0.5, ls="--", color="crimson", lw=1.3)
        axes[1].text(0.52, 0.3, "memorisation risk", color="crimson",
                     fontsize=7, rotation=90, va="bottom")
        axes[1].set_yticks(range(len(dsort)), dsort.index, fontsize=7)
        axes[1].set_title("Privacy — median distance to closest real patient\n"
                          "(higher = less copying)")
        save(fig, 7, "fidelity_privacy")

    # ---- 8. ROC curves ------------------------------------------------------
    if out.curves:
        clf_names = sorted({c for v in out.curves.values() for c in v})
        fig, axes = plt.subplots(1, len(clf_names),
                                 figsize=(5.0 * len(clf_names), 4.6))
        axes = np.atleast_1d(axes)
        for ax, clf in zip(axes, clf_names):
            for gname, cur in out.curves.items():
                if clf in cur:
                    fpr, tpr, auc = cur[clf]
                    ax.plot(fpr, tpr, lw=1.7, color=colour_for(gname),
                            label=f"{gname} ({auc:.3f})",
                            ls="--" if gname.startswith("None") else "-")
            ax.plot([0, 1], [0, 1], color="grey", lw=0.8, ls=":")
            ax.set_title(f"ROC — {clf}")
            ax.set_xlabel("false positive rate")
            ax.set_ylabel("true positive rate")
            ax.legend(fontsize=6, loc="lower right")
        fig.suptitle(f"ROC curves on real test data — {out.dataset_name}",
                     y=1.02, fontweight="bold")
        save(fig, 8, "roc_curves")

    # ---- 9. cost ------------------------------------------------------------
    secs = out.utility.groupby("Generator")["Seconds"].mean().sort_values()
    secs = secs[secs > 0]
    if len(secs):
        fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(secs) + 1.8))
        ax.barh(range(len(secs)), secs.values,
                color=[colour_for(g) for g in secs.index])
        for i, v in enumerate(secs.values):
            ax.text(v * 1.02, i, f"{v:.1f}s", va="center", fontsize=7)
        ax.set_yticks(range(len(secs)), secs.index, fontsize=8)
        ax.set_xlabel("mean fit + sample time (seconds)")
        ax.set_title(f"Computational cost — {out.dataset_name}")
        save(fig, 9, "runtime")

    # ---- 10. final composite ranking ---------------------------------------
    fig, ax = plt.subplots(figsize=(8.5, 0.45 * len(ranking) + 2))
    r = ranking.sort_values("CompositeScore")
    bars = ax.barh(range(len(r)), r["CompositeScore"],
                   color=[colour_for(g) for g in r.index])
    bars[-1].set_edgecolor("gold")
    bars[-1].set_linewidth(2.5)
    for i, v in enumerate(r["CompositeScore"]):
        ax.text(v + 0.008, i, f"{v:.3f}", va="center", fontsize=8)
    ax.set_yticks(range(len(r)), r.index, fontsize=8)
    ax.set_xlabel("composite score  (utility 60% · fidelity 30% · privacy 10%)")
    ax.set_title(f"FINAL RANKING — {out.dataset_name}\n"
                 f"winner: {ranking.index[0]}", fontsize=11)
    save(fig, 10, "final_ranking")

    return saved


# =============================================================================
#  SECTION 6 — COMPOSITE RANKING AND VERDICT
# =============================================================================

W_UTILITY, W_FIDELITY, W_PRIVACY = 0.60, 0.30, 0.10


def rank_generators(out: BenchmarkOutput) -> pd.DataFrame:
    """
    Composite score. Each component is min-max normalised across generators,
    so the score is a RELATIVE ranking within this run -- not an absolute
    quality measure comparable across datasets.

      utility  60%  = 0.35*F1 + 0.15*ROC-AUC + 0.10*PR-AUC   (higher better)
      fidelity 30%  = 0.12*(1-KS) + 0.09*(1-TVD) + 0.09*(1-CorrDiff)
      privacy  10%  = DCR, capped -- beyond "clearly not copying" no extra credit
    """
    u = out.utility.groupby("Generator")[["F1", "ROC-AUC", "PR-AUC"]].mean()

    if not out.fidelity.empty:
        f = out.fidelity.groupby("Generator")[
            ["KS_mean", "TVD_mean", "CorrDiff", "DCR_median"]].mean()
    else:
        f = pd.DataFrame(index=u.index)

    df = u.join(f, how="left")

    def norm(s, higher_better=True):
        s = s.astype(float)
        if s.notna().sum() == 0 or np.nanmax(s) == np.nanmin(s):
            return pd.Series(0.5, index=s.index)
        z = (s - np.nanmin(s)) / (np.nanmax(s) - np.nanmin(s))
        return z if higher_better else 1 - z

    score = (0.35 * norm(df["F1"])
             + 0.15 * norm(df["ROC-AUC"])
             + 0.10 * norm(df["PR-AUC"]))

    if "KS_mean" in df:
        # Baseline has no synthetic data, so no fidelity -> give it the median
        fid = (0.12 * norm(df["KS_mean"], False)
               + 0.09 * norm(df["TVD_mean"], False)
               + 0.09 * norm(df["CorrDiff"], False))
        score = score + fid.fillna(fid.median())

        dcr = df["DCR_median"].clip(upper=df["DCR_median"].median() * 2)
        priv = 0.10 * norm(dcr)
        score = score + priv.fillna(priv.median())

    df["CompositeScore"] = score
    return df.sort_values("CompositeScore", ascending=False)


def print_verdict(out: BenchmarkOutput, ranking: pd.DataFrame):
    pd.set_option("display.width", 220, "display.max_columns", 50)
    name = out.dataset_name
    u = out.utility

    print(f"\n{'=' * 78}\n  RESULTS — {name}\n{'=' * 78}")

    print("\n[1] UTILITY (mean ± std over repeats, real held-out test)")
    mean = u.groupby("Generator")[
        ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]].mean()
    std = u.groupby("Generator")["F1"].std()
    tbl = mean.round(3)
    tbl["F1_std"] = std.round(3)
    print(tbl.sort_values("F1", ascending=False).to_string())

    if not out.fidelity.empty:
        print("\n[2] FIDELITY + PRIVACY  (KS/TVD/CorrDiff lower better; DCR higher better)")
        print(out.fidelity.groupby("Generator")[
            ["KS_mean", "TVD_mean", "CorrDiff", "DCR_median"]
        ].mean().round(3).to_string())

    print(f"\n[3] COMPOSITE RANKING "
          f"(utility {W_UTILITY:.0%} · fidelity {W_FIDELITY:.0%} · privacy {W_PRIVACY:.0%})")
    print(ranking[["F1", "ROC-AUC", "KS_mean", "DCR_median", "CompositeScore"]]
          .round(3).to_string())

    # ---- the verdict --------------------------------------------------------
    best = ranking.index[0]
    runner = ranking.index[1] if len(ranking) > 1 else None
    base_f1 = mean.loc["None (baseline)", "F1"] if "None (baseline)" in mean.index else np.nan
    best_f1 = mean.loc[best, "F1"]
    gap = ranking["CompositeScore"].iloc[0] - ranking["CompositeScore"].iloc[1]
    f1_std = std.get(best, np.nan)

    print(f"\n{'-' * 78}\n  VERDICT — {name}\n{'-' * 78}")
    print(f"  BEST GENERATOR : {best}")
    print(f"  runner-up      : {runner}   (composite gap {gap:.3f})")
    print(f"  F1 vs baseline : {best_f1:.3f} vs {base_f1:.3f} "
          f"({best_f1 - base_f1:+.3f})")
    print(f"  F1 std across {u['Run'].nunique()} splits: ±{f1_std:.3f}")

    if not np.isnan(f1_std) and gap < 0.05:
        print("\n  ! The top two are within 0.05 composite. Treat this as a tie,")
        print("    not a clear winner. Increase --repeats before publishing it.")
    if not np.isnan(f1_std) and abs(best_f1 - base_f1) < f1_std:
        print("\n  ! The improvement over no-augmentation is smaller than the")
        print("    spread across splits. Report it as 'no significant gain'.")

    print()


# =============================================================================
#  SECTION 7 — MAIN
# =============================================================================

def analyse(path, loader, label, args):
    X, y, cats = loader(path)
    out = run_benchmark(X, y, cats, label, repeats=args.repeats,
                        epochs=args.epochs, dl_epochs=args.dl_epochs,
                        cuda=args.cuda)
    ranking = rank_generators(out)
    print_verdict(out, ranking)

    slug = label.split()[0].lower()
    out.utility.to_csv(f"{args.out_prefix}_{slug}_utility.csv", index=False)
    out.fidelity.to_csv(f"{args.out_prefix}_{slug}_fidelity.csv", index=False)
    ranking.to_csv(f"{args.out_prefix}_{slug}_ranking.csv")

    if not args.no_plots:
        paths = make_all_plots(out, ranking, args.plot_dir)
        print(f"  {len(paths)} charts written to '{args.plot_dir}/'")
    return out, ranking


def main():
    ap = argparse.ArgumentParser(
        description="Generative AI benchmark + visualisation on PCOS data")
    ap.add_argument("--clinical",
                    default="/content/drive/MyDrive/PCOD/Training data.csv")
    ap.add_argument("--survey",
                    default="/content/drive/MyDrive/PCOD/Testing data.csv")
    ap.add_argument("--dataset", choices=["clinical", "survey", "both"],
                    default="both")
    ap.add_argument("--repeats", type=int, default=3,
                    help="number of random train/test splits to average over")
    ap.add_argument("--epochs", type=int, default=300, help="CTGAN/TVAE epochs")
    ap.add_argument("--dl-epochs", type=int, default=4000,
                    help="diffusion training steps")
    ap.add_argument("--cuda", action="store_true",
                    help="use GPU for CTGAN/TVAE (Colab: enable a T4 first)")
    ap.add_argument("--plot-dir", default="plots")
    ap.add_argument("--out-prefix", default="pcos_results")
    ap.add_argument("--no-plots", action="store_true")

    # parse_known_args, not parse_args: survives being run inside a Jupyter or
    # Colab cell, where the kernel injects its own "-f kernel.json"
    args, _ = ap.parse_known_args()

    results = {}
    if args.dataset in ("clinical", "both"):
        results["clinical"] = analyse(args.clinical, load_clinical,
                                      "clinical PCOS", args)
    if args.dataset in ("survey", "both"):
        results["survey"] = analyse(args.survey, load_survey,
                                    "survey PCOS", args)

    if len(results) == 2:
        print(f"\n{'=' * 78}\n  CROSS-DATASET SUMMARY\n{'=' * 78}")
        for k, (_, rank) in results.items():
            print(f"  {k:<10} winner: {rank.index[0]:<32} "
                  f"composite {rank['CompositeScore'].iloc[0]:.3f}")
        print("\n  If the two datasets disagree, that is a real finding, not a")
        print("  bug: they have different sizes, feature types and imbalance.")

    print(f"\nDone. CSVs written with prefix '{args.out_prefix}_'.")
    return results


if __name__ == "__main__":
    main()


  clinical PCOS   (3 repeated splits)
  train (405, 41)  test (136, 41)  minority rows to generate: 139
    run 0  [SMOTE                             ]  139 rows    0.0s
    run 0  [ADASYN                            ]  139 rows    0.0s
    run 0  [BorderlineSMOTE                   ]  139 rows    0.0s
    run 0  [CTGAN                             ]  139 rows   49.4s
    run 0  [TVAE                              ]  139 rows   17.2s
    run 0  [CTAB-GAN+ (unavailable -> CTGAN)  ]  139 rows   48.0s
    run 0  [TabDDPM-lite                      ]  139 rows   43.9s
    run 0  [TabSyn-lite                       ]  139 rows   54.8s
    run 1  [SMOTE                             ]  139 rows    0.0s
    run 1  [ADASYN                            ]  139 rows    0.0s
    run 1  [BorderlineSMOTE                   ]  139 rows    0.0s
    run 1  [CTGAN                             ]  139 rows   47.0s
    run 1  [TVAE                              ]  139 rows   15.6s
    run 1  [CTAB-GAN+ (unavailable ->